
# Weather Disruption Score (WDS) — Focused EDA Notebook
This notebook loads **all cleaned airport-year CSVs** and performs *operationally meaningful* EDA geared toward creating a **1–100 Weather Disruption Score**.
We intentionally **focus on extreme/risky conditions** instead of raw distributions that are dominated by clear weather.



**What you'll get here:**
- Robust loader (airport from filename if needed)
- Severity flags (IFR/LIFR thresholds, wind/precip cutoffs)
- Extreme-weather filtering & plots (inline)
- Joint analyses (ceiling vs visibility in risky regimes)
- Volatility features (hour-to-hour changes)
- Clustering (weather regimes) and PCA (2D projection)
- A prototype `proto_score` (0–100) for experimentation

> Tip: Run the setup cell first to ensure packages are available.


In [ ]:

# --- Setup: install/imports (safe re-run) ---
import sys, subprocess

def _pip(pkg):
    try:
        __import__(pkg)
    except Exception:
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

for pkg in ["pandas", "numpy", "matplotlib", "scikit-learn"]:
    name = "scikit-learn" if pkg=="scikit-learn" else pkg
    _pip(name)

import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["figure.dpi"] = 120
print("Ready.")


## Paths

In [ ]:

# Adjust these if your folders are different
CLEAN_DIR = "data_clean"
GLOB = "*_clean.csv"

import glob, os
files = sorted(glob.glob(os.path.join(CLEAN_DIR, GLOB)))
len(files), files[:5]


## Load all cleaned files

In [ ]:

import os, glob, pandas as pd
def load_all(clean_dir: str, pattern: str = "*_clean.csv") -> pd.DataFrame:
    files = sorted(glob.glob(os.path.join(clean_dir, pattern)))
    if not files:
        raise FileNotFoundError(f"No files matched: {clean_dir}/{pattern}")
    dfs = []
    for fp in files:
        df = pd.read_csv(fp, parse_dates=["datetime"])
        # Prefer existing airport column, else derive from filename
        if "airport" in df.columns and df["airport"].notna().any():
            airport_code = str(df["airport"].dropna().iloc[0])
        else:
            airport_code = os.path.basename(fp).split("_")[0]
        df["airport"] = airport_code

        if "year" not in df.columns:
            df["year"] = df["datetime"].dt.year
        dfs.append(df)
    big = pd.concat(dfs, ignore_index=True)
    big.sort_values(["airport", "datetime"], inplace=True)
    big.reset_index(drop=True, inplace=True)
    return big

df = load_all(CLEAN_DIR, GLOB)
df.shape, df["airport"].nunique(), df.head(3)


## Severity flags & helpers (IFR/LIFR thresholds, wind/precip cutoffs)

In [ ]:

def add_severity_flags(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["sev_low_vis"]          = df["vis_m"] < 1600
    df["sev_very_low_vis"]     = df["vis_m"] < 600
    df["sev_low_ceiling"]      = df["ceiling_m"] < 300
    df["sev_very_low_ceiling"] = df["ceiling_m"] < 100
    df["sev_high_wind"]        = df["wind_speed_ms"] > 15
    df["sev_very_high_wind"]   = df["wind_speed_ms"] > 20
    df["sev_heavy_rain"]       = df["precip_mm_1h"] > 5

    df["dew_spread"] = df["temp_c"] - df["dewpoint_c"]
    df["sev_fog"]    = (df["vis_m"] < 1000) & (df["dew_spread"] < 2)

    df["month"] = df["datetime"].dt.month
    df["hour"]  = df["datetime"].dt.hour
    return df

df = add_severity_flags(df)
df[[c for c in df.columns if c.startswith("sev_")]].mean().sort_values(ascending=False)


## Focused dataset: filter to 'interesting' weather only

In [ ]:

risk_mask = (
    (df["vis_m"] < 5000) |
    (df["ceiling_m"] < 1000) |
    (df["wind_speed_ms"] > 8) |
    (df["precip_mm_1h"] > 1)
)
df_risk = df.loc[risk_mask].copy()
df_risk.shape, df.shape


## Distributions *only within risky subset*

In [ ]:

def hist(series, title, vlines=None, bins=50):
    s = series.dropna().values
    fig, ax = plt.subplots()
    ax.hist(s, bins=bins)
    if vlines:
        for v in vlines:
            ax.axvline(v, linestyle="--")
    ax.set_title(title)
    ax.set_xlabel(series.name)
    ax.set_ylabel("count")
    plt.show()

hist(df_risk["vis_m"], "Visibility (risky subset)", vlines=[1600, 600])
hist(df_risk["ceiling_m"], "Ceiling (risky subset)", vlines=[300, 100])
hist(df_risk["wind_speed_ms"], "Wind speed (risky subset)", vlines=[10, 15, 20])
hist(df_risk["precip_mm_1h"], "Precip intensity (risky subset)", vlines=[2, 5, 15])


## Joint analysis (ceiling vs visibility) — risky subset

In [ ]:

def scatter(df_, x, y, title, sample=60000):
    d = df_[[x,y]].dropna()
    if sample and len(d) > sample:
        d = d.sample(sample, random_state=0)
    fig, ax = plt.subplots()
    ax.scatter(d[x].values, d[y].values, alpha=0.3)
    ax.set_title(title)
    ax.set_xlabel(x); ax.set_ylabel(y)
    plt.show()

scatter(df_risk, "ceiling_m", "vis_m", "Ceiling vs Visibility (risky subset)")


## Volatility features (hour-to-hour deltas)

In [ ]:

df["wind_change"] = df.groupby("airport")["wind_speed_ms"].diff()
df["temp_change"] = df.groupby("airport")["temp_c"].diff()
df["vis_change"]  = df.groupby("airport")["vis_m"].diff()

df[["wind_change","temp_change","vis_change"]].describe()


In [ ]:

hist(df_risk["wind_change"], "Δ Wind speed (risky subset)", vlines=[2, 4])
hist(df_risk["vis_change"], "Δ Visibility (risky subset)", vlines=[-2000, -1000])


## Weather regimes (KMeans on risky subset)

In [ ]:

features = ["wind_speed_ms","vis_m","ceiling_m","temp_c","precip_mm_1h"]
work = df_risk[features].copy()
all_na = work.isna().all(axis=1)
work = work.loc[~all_na]
fill_vals = work.median(numeric_only=True)
work = work.fillna(fill_vals)

if len(work) > 150_000:
    work_sample = work.sample(150_000, random_state=0)
else:
    work_sample = work

scaler = StandardScaler()
X = scaler.fit_transform(work_sample.values)

kmeans = KMeans(n_clusters=8, random_state=0, n_init="auto")
labels = kmeans.fit_predict(X)
centers = scaler.inverse_transform(kmeans.cluster_centers_)

centers_df = pd.DataFrame(centers, columns=features).sort_values("vis_m")
centers_df


In [ ]:

pca = PCA(n_components=2, random_state=0)
pcs = pca.fit_transform(X)
fig, ax = plt.subplots()
ax.scatter(pcs[:,0], pcs[:,1], alpha=0.2, s=3)
ax.set_title("Risky subset projected to 2D (PCA)")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.show()


## Prototype 0–100 score

In [ ]:

def prototype_score(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    w = {
        "sev_low_vis": 20,
        "sev_very_low_vis": 25,
        "sev_low_ceiling": 20,
        "sev_very_low_ceiling": 25,
        "sev_high_wind": 10,
        "sev_heavy_rain": 10,
        "sev_fog": 30
    }
    for k in w:
        if k not in df.columns:
            df[k] = False
    raw = (
        w["sev_low_vis"]          * df["sev_low_vis"].astype(int) +
        w["sev_very_low_vis"]     * df["sev_very_low_vis"].astype(int) +
        w["sev_low_ceiling"]      * df["sev_low_ceiling"].astype(int) +
        w["sev_very_low_ceiling"] * df["sev_very_low_ceiling"].astype(int) +
        w["sev_high_wind"]        * df["sev_high_wind"].astype(int) +
        w["sev_heavy_rain"]       * df["sev_heavy_rain"].astype(int) +
        w["sev_fog"]              * df["sev_fog"].astype(int)
    )
    raw_max = raw.max() if raw.max() > 0 else 1.0
    df["proto_score"] = (raw / raw_max) * 100.0
    return df

df = prototype_score(df)
df["proto_score"].describe()


In [ ]:

# Daily mean prototype score for first few airports
for airport in df["airport"].dropna().unique()[:6]:
    sub = df.loc[df["airport"] == airport].copy()
    if len(sub) == 0:
        continue
    daily = sub.resample("1D", on="datetime")["proto_score"].mean().reset_index()
    fig, ax = plt.subplots()
    ax.plot(daily["datetime"].values, daily["proto_score"].values)
    ax.set_title(f"{airport} — Daily mean prototype score")
    ax.set_xlabel("date"); ax.set_ylabel("proto_score")
    plt.show()
